# Lab 2 - Part B : Inverse Kinematics  (answer key)

RRR robot: reach the target M with the hand-solved q, verify with FK, check every branch.

## 0. Setup

In [ ]:
import numpy as np
from pathlib import Path
from roboticstoolbox import Robot
from spatialmath import SE3
import spatialgeometry as sg
import swift

HERE = Path.cwd().as_posix()
links, name, _, _ = Robot.URDF_read('robot/robot.urdf', tld=HERE)
robot = Robot(links, name=name)
print('loaded', name, '|', robot.n, 'joints')

env = swift.Swift()
env.launch(realtime=True, browser="notebook")   # inline in the cell (like Lab 1)
env.add(robot)
robot.q = [0.0, 0.0, 0.0]   
env.step()

## 1. Target M and the hand configuration

In [ ]:
# 1.1  target M
M      = np.array([0.3890, -0.2902, 0.3690])   # target (base frame)
q_goal = np.array([-0.6000, 0.6000, -0.5000])   # hand-solved IK

marker = sg.Sphere(radius=0.02, color=(1.0, 0.1, 0.1, 0.6))
marker.T = SE3(*M)
env.add(marker)
env.step()

In [ ]:
# 1.2  move the arm to q_goal
robot.q = q_goal
env.step()
print('tip:', np.round(robot.fkine(q_goal).t, 4))

## 2. Cross-check with FK

In [ ]:
tip = robot.fkine(q_goal).t
err = np.linalg.norm(tip - M)
print('tip   :', np.round(tip, 4))
print('target:', np.round(M, 4))
print('error :', round(float(err), 4), 'm')

## 3. Solve IK with the toolbox (ikine_LM) and compare

In [ ]:
sol = robot.ikine_LM(SE3(*M), q0=[0, 0, 0], mask=[1, 1, 1, 0, 0, 0], joint_limits=True)  # mask = position only
print('toolbox q:', np.round(sol.q, 4), '| ok', sol.success)
print('hand q   :', np.round(q_goal, 4))

## 4. Check every branch
This RRR robot has two elbow configurations that both reach M.

In [ ]:
branches = [
    np.array([-0.6000, 0.6004, -0.5008]),
    np.array([-0.6000, 0.0708, 0.5005]),
]

for i, qb in enumerate(branches):
    err = np.linalg.norm(robot.fkine(qb).t - M)
    print('branch', i + 1, np.round(qb, 4), '-> err', round(float(err), 4))